# Causal feature-engineered CNN + LSTM
An opt-in experiment based on `training_lstm.ipynb`: the same causal CNN, LSTM, classifier, delayed targets, positive-window augmentation, loss, AdamW optimizer, cosine schedule, and window-based checkpoint selection. The input projection receives engineered features instead of just the six IMU axes. Training starts from scratch.

The model still accepts `[batch, time, 6]` in the order `acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z`. Features are computed **inside the model**, after the dataset's high-pass filter and any raw-axis amplitude augmentation. No features use labels, future frames, validation statistics, or whole-recording normalization.

Open with the repository `.venv` kernel and run in order. Every training invocation creates a new directory under `testing_checkpoints/lstm64_features/`. The data and existing model runs are read-only inputs.

In [ ]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'tap_recognition').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'tap_recognition').is_dir(), 'Open from repository root or main_notebooks/'
sys.path.insert(0, str(ROOT))
from tap_recognition import model_factory
importlib.reload(model_factory)
from tap_recognition.config import DataConfig, EarlyStoppingConfig, LSTMModelConfig, TrainConfig, TrainingConfig
from tap_recognition.imu_features import CausalIMUFeatures, IMUFeatureConfig
from tap_recognition.feature_training import build_feature_datasets, train_feature_lstm
torch.set_num_threads(2)


## Experiment settings
Keep seed, delay, split, augmentation, and optimizer settings matched to your raw-axis baseline. The current defaults use LSTM64, one layer, delay 10, seed 101, and the expanded split. Choose seed 102 in a separate invocation to examine training-seed variation.

Default **20 channels**: six signed axes; acceleration/gyro magnitudes; six per-sample differences; two difference magnitudes; and acceleration/gyro trailing RMS at 5 and 15 samples (50/150 ms at 100 Hz). Signed axes preserve left/right information. RMS is the square root of trailing mean squared vector magnitude, with zero prehistory. The first difference at a reset is zero. Disable groups or change `rms_windows` for ablations.

Training-only per-channel mean/std normalization gives the extra channels comparable scales. Statistics are fitted once on the unaugmented training windows (including their existing padding), frozen afterwards, and saved with the model weights.

In [ ]:
DATA_ROOT = ROOT / 'data_05_10_26'
TRAINING_SEED = 101
LABEL_DELAY_FRAMES = 10
AMPLITUDE_AUGMENTATION = True
FEATURES = IMUFeatureConfig(
    magnitudes=True,
    differences=True,
    difference_magnitudes=True,
    rms_windows=(5, 15),
)
cfg = TrainConfig(
    seed=TRAINING_SEED,
    device='auto',
    out_dir=str(ROOT / 'testing_checkpoints/lstm64_features'),
    data=DataConfig(
        train_dir=str(DATA_ROOT / 'train_data'),
        val_dir=str(DATA_ROOT / 'valid_data'),
        session_exclusions_file=str(DATA_ROOT / 'session_exclusions.csv'),
        highpass=True,
    ),
    model=LSTMModelConfig(lstm_hidden=64, lstm_layers=1),
    training=TrainingConfig(
        epochs=60, batch_size=16, lr=1e-3,
        early_stopping=EarlyStoppingConfig(monitor='val_window_acc'),
    ),
)
for name in ('train_data', 'valid_data'):
    if not (DATA_ROOT / name).is_dir():
        raise FileNotFoundError(DATA_ROOT / name)
if not Path(cfg.data.session_exclusions_file).is_file():
    raise FileNotFoundError(cfg.data.session_exclusions_file)
print(json.dumps(cfg.to_dict(), indent=2))
display(pd.DataFrame({'index': range(len(FEATURES.names)), 'feature': FEATURES.names}))


## Load and inspect the explicit split
This uses the same recorded-window loader as the baseline and the selected split's exclusion registry. An invalid validation split is reported rather than replaced by a random subset of training. No output directory has been created yet.

In [ ]:
train_ds, val_ds = build_feature_datasets(cfg)
balance = []
for split, dataset in [('train', train_ds), ('valid', val_ds)]:
    counts = pd.Series([int(item['window_label']) for item in dataset]).value_counts()
    balance.append({'split': split, 'windows': len(dataset),
                    'none': int(counts.get(0, 0)), 'left': int(counts.get(1, 0)),
                    'right': int(counts.get(2, 0))})
display(pd.DataFrame(balance))
preview = CausalIMUFeatures(FEATURES)(train_ds[0]['imu'].unsqueeze(0))
assert torch.isfinite(preview).all()
print('Raw shape:', tuple(train_ds[0]['imu'].shape), '| Engineered shape:', tuple(preview.shape))


## Train a new run
The next cell trains for up to `cfg.training.epochs` epochs. Set it to 1 for a quick smoke experiment. Re-executing this cell starts another independent run in a fresh timestamped directory.

Artifacts: `initial.pt`, `best.pt`, `last.pt`, `history.csv`, `experiment.json`, and `data_manifest.json`. `best.pt` uses the configured validation monitor; `last.pt` and history are updated each completed epoch. The architecture is saved as `model_type='feature_lstm'` with feature settings, channel order, normalization statistics, delay, augmentation, and training seed.

In [ ]:
run_dir, history = train_feature_lstm(
    cfg, FEATURES, train_ds, val_ds,
    label_delay_frames=LABEL_DELAY_FRAMES,
    amplitude_augmentation=AMPLITUDE_AUGMENTATION,
)
display(history.tail())
print(f"In evaluate.ipynb set: RUN_DIR = ROOT / {str(run_dir.relative_to(ROOT))!r}")


## Evaluate and compare
In `evaluate.ipynb`, set `RUN_DIR` to the printed run directory, keep `EVALUATION_DATA_ROOT` on the same prepared split, and use the same `EVALUATION_SEED` and matching rules as the baseline. Restart its kernel once after adding this new architecture so the model factory is current. The factory reconstructs the feature transform and frozen normalization automatically; feed the ordinary six-channel IMU tensors, **not** precomputed features. The overlap-streaming reference includes feature history.

Use the resulting `threshold_comparison/` folder in your comparison notebook. Feature extraction is an architectural difference saved in `model_config`; the evaluation-data fingerprint continues to describe the common raw data and labels. All other data and alarm-policy compatibility checks still apply.

Compare both architectures over the same training seeds. Adding input channels slightly increases projection parameters; there is no assumption that feature engineering will improve accuracy. Window-level training selection remains descriptive validation, not an independent test estimate.